# Falcon 9 expert mission-planning demo

This notebook presents one quantitative expert screening case: a 3,000 kg payload into a 200 km circular low Earth orbit (LEO). It uses the repository's existing mission pipeline and report generator, with deterministic physics calculations and a bounded sensitivity analysis.

The analysis covers vehicle parameters, staging, ascent and transfer delta-v budgets, payload ceiling, reserve assumptions, and limitations. It is a screening calculation, not a flight-design release. Run from the repository root; mission parsing and the agent loop require the `TAMU_API_KEY` described in the README. Generated JSON and reports are saved under `reports/examples/`.

In [12]:
from pathlib import Path
import json
import subprocess
import sys

ROOT = Path.cwd()
if not (ROOT / "src" / "run_mission.py").is_file():
    raise RuntimeError("Run this notebook with the repository root as the working directory.")

ROCKET_SPEC = ROOT / "rockets" / "falcon9.txt"
EXAMPLES_DIR = ROOT / "examples"
REPORTS_DIR = ROOT / "reports" / "examples"
SCENARIOS = [("expert", "Expert", EXAMPLES_DIR / "expert.txt")]

for slug, persona, path in SCENARIOS:
    if not path.is_file():
        raise FileNotFoundError(f"Missing {persona} example input: {path}")
    print(f"{persona}: {path.relative_to(ROOT)}")

Expert: examples/expert.txt


## 1. Expert screening case

The input below specifies the requested payload, orbit, vehicle data, ascent-loss budget, reserve target, and requested outputs. Its limitations explicitly separate modeled results from unsupported fault analysis.

In [13]:
for slug, persona, path in SCENARIOS:
    print(f"\n{'=' * 16} {persona} {'=' * 16}\n")
    print(path.read_text(encoding="utf-8"))


================ Expert ================

# Persona: Expert — an aerospace engineer specifying a screening case
[MISSION REQUEST]
Screen Falcon 9 for a 3,000 kg payload into a 200 km circular low Earth orbit (LEO). Assume nominal sequential staging and use the supplied vehicle data: stage 1 sea-level/vacuum Isp of 283/312 s and stage 2 vacuum Isp of 348 s. Use a 1,500 m/s ascent-loss budget. Report stage propellant mass fractions, a parameter and burn table, liftoff thrust-to-weight, required versus available ascent delta-v, the payload ceiling, and the resulting margin. My design target includes a 5% ascent delta-v reserve; state explicitly whether the repository model applies it. Include a small sensitivity sweep and identify relevant failure cases, distinguishing modeled results from unsupported fault analysis.

[LIMITATIONS]
Use the repository's Falcon 9 data and deterministic model only. This is a screening calculation, not a flight-design release; do not imply human-rating, luna

## 2. Run the pipeline and generate the report

The notebook saves the expert mission result as JSON, then invokes the report generator with deterministic prose and benchmarking disabled. It adds the expert screening annex to the generated Markdown report; all mission calculations remain the pipeline's results.

In [14]:
def run_command(args):
    completed = subprocess.run(args, cwd=ROOT, text=True, capture_output=True, check=False)
    if completed.returncode:
        detail = completed.stderr.strip() or completed.stdout.strip()
        raise RuntimeError(f"Command failed ({completed.returncode}): {' '.join(map(str, args))}\n{detail}")
    return completed.stdout


def fmt(value, digits=1):
    return "not reported" if value is None else f"{value:,.{digits}f}"


def sourced_value(stage, name):
    return (stage.get(name) or {}).get("value")


def append_notes(report_path, result, slug):
    agent = result.get("agent", {})
    facts = agent.get("facts", {})
    ascent = agent.get("ascent", {})
    transfer = agent.get("transfer", {})
    margin = agent.get("margin", {})
    requested = facts.get("requested_kg", result.get("request", {}).get("payload_kg"))
    destination = result.get("request", {}).get("destination", "the requested destination")
    ceiling = facts.get("ceiling_kg", agent.get("max_payload", {}).get("max_payload_kg"))

    from src.ascent_loop.physics import build_rocket, find_max_payload

    rows = []
    rocket_spec = {"name": result.get("rocket", {}).get("name", "Falcon 9"), "stages": []}
    for index, stage in enumerate(result.get("rocket", {}).get("stages", []), start=1):
        dry = sourced_value(stage, "dry_mass_kg")
        prop = sourced_value(stage, "propellant_mass_kg")
        isp_vac = sourced_value(stage, "isp_vac_s")
        isp_sl = sourced_value(stage, "isp_sl_s")
        wet_mass = (dry or 0) + (prop or 0)
        prop_fraction = prop / wet_mass * 100 if wet_mass else None
        rows.append(f"| {index} | {fmt(dry, 0)} | {fmt(prop, 0)} | {fmt(isp_sl, 0)} | {fmt(isp_vac, 0)} | {fmt(prop_fraction, 1)}% |")
        rocket_spec["stages"].append({
            "dry_mass_kg": dry,
            "propellant_mass_kg": prop,
            "isp_vac_s": isp_vac,
            "isp_sl_s": isp_sl,
            "thrust_sl_n": sourced_value(stage, "thrust_sl_n"),
        })
    reserve_dv = margin.get("reserved_transfer_dv_ms", transfer.get("reserved_transfer_dv_ms", transfer.get("dv_total_ms", 0)))
    reserve_fraction = margin.get("reserve_fraction")
    combined_margin = margin.get("margin_ms")
    rocket = build_rocket(rocket_spec)
    sensitivity = []
    for loss_budget in (1000.0, 1500.0, 2000.0):
        ceiling_at_loss = find_max_payload(rocket, losses_ms=loss_budget, required_extra_dv_ms=reserve_dv)["max_payload_kg"]
        sensitivity.append(f"| {loss_budget:,.0f} | {ceiling_at_loss:,.0f} |")
    ascent_reserve_dv = ascent.get("dv_needed_ms", 0) * 0.05
    ceiling_plus_reserve = find_max_payload(rocket, losses_ms=1500.0, required_extra_dv_ms=reserve_dv + ascent_reserve_dv)["max_payload_kg"]
    lines = [
        "## Expert screening annex",
        "",
        "### Vehicle parameters",
        "",
        "| Stage | Dry mass (kg) | Propellant (kg) | Isp, SL (s) | Isp, vacuum (s) | Propellant / wet stage mass |",
        "| ---: | ---: | ---: | ---: | ---: | ---: |",
        *rows,
        "",
        f"Requested payload: **{fmt(requested, 0)} kg to {destination}**; liftoff T/W: **{fmt(ascent.get('liftoff_twr'), 2)}**. The report's burn table contains per-stage mass states and delta-v. Ascent delta-v: **{fmt(ascent.get('dv_total_ms'))} m/s available** versus **{fmt(ascent.get('dv_needed_ms'))} m/s required**; transfer delta-v: **{fmt(transfer.get('dv_total_ms'))} m/s**; applied transfer reserve: **{fmt(reserve_dv)} m/s ({fmt((reserve_fraction or 0) * 100, 1)}%)**; residual combined margin: **{fmt(combined_margin)} m/s**; mission payload ceiling: **{fmt(ceiling, 0)} kg**.",
        "",
        "### Sensitivity: fixed loss budget, all else held constant",
        "",
        "| Ascent-loss budget (m/s) | Mission payload ceiling (kg) |",
        "| ---: | ---: |",
        *sensitivity,
        "",
        f"The pipeline applies a {fmt((reserve_fraction or 0) * 100, 1)}% reserve to modeled transfer delta-v; it does not apply the requested 5% ascent reserve. A separate sensitivity case adding 5% of the required ascent delta-v, on top of the modeled transfer reserve, gives a payload ceiling of **{ceiling_plus_reserve:,.0f} kg**.",
        "",
        "### Failure modes and limitations",
        "",
        "| Case | Treatment |",
        "| --- | --- |",
        "| Ascent/transfer budget shortfall | Quantitatively screened by the saved budget and margin outputs. |",
        "| Low liftoff thrust-to-weight | Nominal check reported; engine degradation/transients not simulated. |",
        "| Engine-out, stage separation, or guidance failure | Not modeled; no fault tree or reliability probabilities are available. |",
        f"| Destination-specific operations | {('ISS rendezvous/docking and plane changes' if destination == 'ISS' else 'lunar capture, landing, and return') if destination in {'ISS', 'Moon', 'TLI'} else 'orbit insertion details beyond the simplified budget'} are outside scope. |",
        "",
        "The deterministic sensitivity sweep varies only the fixed ascent-loss budget while holding vehicle data and transfer reserve constant. Nominal sequential staging is assumed. These results are not uncertainty bounds or a flight-design release.",
    ]
    with report_path.open("a", encoding="utf-8") as report_file:
        report_file.write("\n\n" + "\n".join(lines) + "\n")


results = {}
for slug, persona, input_path in SCENARIOS:
    output_dir = REPORTS_DIR / slug
    output_dir.mkdir(parents=True, exist_ok=True)
    result_path = output_dir / "result.json"
    report_path = output_dir / "report.md"
    request = input_path.read_text(encoding="utf-8")
    raw_json = run_command([sys.executable, "-m", "src.run_mission", request, str(ROCKET_SPEC), "5"])
    result = json.loads(raw_json)
    result_path.write_text(json.dumps(result, indent=2) + "\n", encoding="utf-8")
    run_command([sys.executable, "-m", "src.report", str(result_path), "--out", str(output_dir)])
    append_notes(report_path, result, slug)
    results[slug] = result
    print(f"{persona}: {report_path.relative_to(ROOT)} (JSON: {result_path.relative_to(ROOT)})")

Expert: reports/examples/expert/report.md (JSON: reports/examples/expert/result.json)


## 3. Review the generated expert report

The report contains the pipeline verdict, burn and budget details, assumptions, provenance, and expert screening annex. Inspect those assumptions before interpreting a feasibility result. This estimate is not a flight plan or a claim that the vehicle is suitable for human spaceflight.

In [10]:
for slug, persona, _ in SCENARIOS:
    report_path = REPORTS_DIR / slug / "report.md"
    print(f"{persona}: {report_path.relative_to(ROOT)} — {report_path.stat().st_size:,} bytes")

Expert: reports/examples/expert/report.md — 5,130 bytes
